<a href="https://colab.research.google.com/github/MFahrezaF/inaportnet-vessel-report-scraper/blob/main/inaportnet_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install playwright pandas openpyxl lxml beautifulsoup4 tqdm nest_asyncio
!playwright install chromium
!playwright install-deps chromium

In [ ]:
BULAN_LIST = [1]
TAHUN      = 2026
DELAY      = 1.0
OUTPUT_DIR = "output_lk3_ln"
BASE_URL   = "https://monitoring-inaportnet.dephub.go.id"

In [ ]:
import os, re, asyncio, shutil
from io import StringIO, BytesIO
import nest_asyncio; nest_asyncio.apply()
import pandas as pd
from tqdm.auto import tqdm
from playwright.async_api import async_playwright

os.makedirs(OUTPUT_DIR, exist_ok=True)

JS_COLLECT = r"""
() => [...document.querySelectorAll('tr')].map(tr => {
  const tds = [...tr.querySelectorAll('td')];
  const a = [...tr.querySelectorAll('a')].find(x => /LK3\s*LN/i.test(x.textContent));
  return a ? {nama: ((tds[1] || {}).textContent || '').trim(), href: a.href} : null;
}).filter(Boolean)
"""

async def kumpulkan_link(page, bulan):
    """Buka main page, then ambil nama pelabuhan + link btn LK3 LN (semua halaman tabel)."""
    await page.goto(f"{BASE_URL}/?bulan={bulan}&tahun={TAHUN}", wait_until="networkidle", timeout=90000)
    await page.wait_for_selector("table tbody tr td", timeout=90000)

    try:
        await page.evaluate("() => $('table').DataTable().page.len(-1).draw()")
        await page.wait_for_timeout(800)
    except Exception:
        pass

    hasil = {}
    for _ in range(500):
        for r in await page.evaluate(JS_COLLECT):
            hasil[r["href"]] = r["nama"]
        nxt = page.locator(".paginate_button.next")
        if await nxt.count() == 0:
            break
        if "disabled" in (await nxt.first.get_attribute("class") or ""):
            break
        await nxt.first.click()
        await page.wait_for_timeout(300)

    if not hasil:
        await page.screenshot(path="debug.png", full_page=True)
        open("debug.html", "w", encoding="utf-8").write(await page.content())
        raise RuntimeError("Tidak ada tombol LK3 LN ditemukan. Lihat debug.png / debug.html.")
    return [{"nama": n, "href": h} for h, n in hasil.items()]

def ambil_kode(href, nama):
    m = re.search(r"ID[A-Z0-9]{3}", href)
    return m.group(0) if m else re.sub(r"\W+", "_", nama)

def parse_file(body, nama, kode):
    df = None
    try:
        tabel = pd.read_html(StringIO(body.decode("utf-8", errors="replace")))
        df = max(tabel, key=len)
    except Exception:
        try:
            df = pd.read_excel(BytesIO(body))
        except Exception:
            return None
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join(str(c) for c in col if "Unnamed" not in str(c)).strip() for col in df.columns]
    df.insert(0, "KODE_PELABUHAN", kode)
    df.insert(0, "PELABUHAN", nama)
    return df

async def proses_bulan(context, page, bulan):
    links = await kumpulkan_link(page, bulan)
    print(f"[Bulan {bulan}] {len(links)} pelabuhan ditemukan. Contoh link: {links[0]['href']}")
    folder = os.path.join(OUTPUT_DIR, f"{TAHUN}_{bulan:02d}")
    os.makedirs(folder, exist_ok=True)

    dfs, gagal = [], []
    for p in tqdm(links, desc=f"Bulan {bulan}"):
        kode = ambil_kode(p["href"], p["nama"])
        path = os.path.join(folder, f"LK3_LN_{kode}_{TAHUN}_{bulan:02d}.xls")
        if os.path.exists(path):
            body = open(path, "rb").read()
        else:
            try:
                resp = await context.request.get(p["href"], timeout=90000)
                body = await resp.body() if resp.ok else None
            except Exception:
                body = None
            if not body or len(body) < 300:
                gagal.append(p["nama"]); continue
            open(path, "wb").write(body)
            await asyncio.sleep(DELAY)
        df = parse_file(body, p["nama"], kode)
        if df is not None and not df.empty:
            dfs.append(df)

    if dfs:
        out = os.path.join(OUTPUT_DIR, f"LK3_LN_SEMUA_{TAHUN}_{bulan:02d}.xlsx")
        pd.concat(dfs, ignore_index=True).to_excel(out, index=False)
        print(f"  Gabungan: {out}")
    print(f"  Berhasil: {len(links) - len(gagal)} | Gagal/kosong: {len(gagal)}", gagal[:10])

async def main():
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True, args=["--no-sandbox"])
        context = await browser.new_context()
        page = await context.new_page()
        for b in BULAN_LIST:
            await proses_bulan(context, page, b)
        await browser.close()

In [ ]:
await main()

In [ ]:
shutil.make_archive("hasil_lk3_ln", "zip", OUTPUT_DIR)
from google.colab import files
files.download("hasil_lk3_ln.zip")